In [8]:
%load_ext autoreload 
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import polars as pl
import datetime as dt

import numpy as np


### Before we get started, let's prototype the functional behaviour we want on a simpler problem

In [10]:
from typing import Self

class Operation:
    def __init__(self, nodes: list[Self]):
        self.nodes = nodes

    def operate(self, x):
        current = x
        for node in self.nodes:
            current = node.operate(current)
        return current

    def compose(self, other: Self):
        return Operation(self.nodes + other.nodes)

    def __sum__(self, other: Self):
        return self.compose(other)

class PrimitiveOperation(Operation):

    def __init__(self):
        self.nodes = [self]


class Add2(PrimitiveOperation):

    def operate(self, x):
        return x + 2


In [11]:
add2 = Add2()
add4 = add2.compose(add2)

In [12]:
add4.operate(1)

5

### Let's start prototyping a new PoMap

### Load the Logic from the repo instead

In [13]:
from categorical_pomap import CategoricalPomap

#### Example time

In [14]:
example_df = pl.from_dict(
    {'cat1': ['a',  'b',],
    'cat2': ['c', 'd', ]}
)

timestamps = pl.datetime_range(dt.datetime(2025, 1, 1),
                               dt.datetime(2025, 1, 7),
                               interval='1h',
                               eager=True
                               ).rename('ts').to_frame()

example_df = example_df.join(timestamps, how='cross')
example_df = example_df.with_columns(
    feature = np.random.normal(0, 1, example_df.shape[0])
)

example_df = example_df.with_columns(hour=pl.col('ts').dt.hour())

In [15]:
pmap_cat1 = CategoricalPomap(column='cat1', labels=['a', 'b'])

In [16]:
pmap_cat2 = CategoricalPomap(column='cat2', labels=['c', 'd'])

In [17]:
composed = pmap_cat1.product(pmap_cat2)

In [18]:
composed.labels

cat1,cat2
str,str
"""a""","""c"""
"""a""","""d"""
"""b""","""c"""
"""b""","""d"""


In [19]:
# Test labeling logic

In [20]:
pmap_cat1.label_rows_as_train(example_df, label={'cat1': 'a'}).head()

cat1,cat2,ts,feature,hour,train({'cat1': 'a'})
str,str,datetime[μs],f64,i8,bool
"""a""","""c""",2025-01-01 00:00:00,0.302955,0,true
"""a""","""c""",2025-01-01 01:00:00,0.03818,1,true
"""a""","""c""",2025-01-01 02:00:00,2.497519,2,true
"""a""","""c""",2025-01-01 03:00:00,2.174522,3,true
"""a""","""c""",2025-01-01 04:00:00,1.447371,4,true


In [21]:
composed = pmap_cat1.product(pmap_cat2)
composed.label_rows_as_train(example_df, label={'cat1': 'a', 'cat2': 'c'}).head()

cat1,cat2,ts,feature,hour,"train({'cat1': 'a', 'cat2': 'c'})"
str,str,datetime[μs],f64,i8,bool
"""a""","""c""",2025-01-01 00:00:00,0.302955,0,true
"""a""","""c""",2025-01-01 01:00:00,0.03818,1,true
"""a""","""c""",2025-01-01 02:00:00,2.497519,2,true
"""a""","""c""",2025-01-01 03:00:00,2.174522,3,true
"""a""","""c""",2025-01-01 04:00:00,1.447371,4,true


### Prototype the model behaviour

In [24]:
import polars as pl
import statsmodels.formula.api as smf

from pomap.pomap import Pomap


In [25]:
#### Toy example

N = 1000
x = np.random.normal(0, 1, size=N)
eps = np.random.normal(scale=0.1, size=x.shape[0])

df = pl.from_dict({'x': x, 'eps': eps})

df = df.with_columns(
    cat1=np.random.choice(['a', 'b'], size=df.shape[0]),
    cat2=np.random.choice(['c', 'd'], size=df.shape[0])
                    )

df = df.with_columns(slope = pl.col('cat1').replace_strict({'a': 5, 'b': 50}),
                     intercept = pl.col('cat2').replace_strict({'c': 0, 'd': -100})
                    )

df = df.with_columns(y=pl.col('slope')*pl.col('x') + pl.col('intercept') + pl.col('eps'))

# We don't get to see eps or the model parameters
train_df = df.select('x', 'y', 'cat1', 'cat2')

In [26]:
constructor = lambda train_data: smf.ols(formula='y~x', data=train_data.to_pandas())

pomap = CategoricalPomap('cat1', ['a', 'b']).product(
    CategoricalPomap('cat2', ['c',])
)

In [27]:
# Need to implement this in PoMap, so that it expands on composition
from collections import namedtuple
Label = namedtuple('Label', ['cat1', 'cat2'])

In [28]:
class LiftedSmf:

    def __init__(self, model_constructor, pomap: Pomap):
        self._constructor = model_constructor
        self.pomap = pomap
        self.models = {}
        self.Label = namedtuple('Label', self.pomap.labels.columns)

    def fit(self, train_data: pl.DataFrame):

        # TODO replace with iter_labels
        for label in self.pomap.labels.iter_rows(named=True):
            sub = self.pomap.label_to_train(df, label=label)
            model = self._constructor(sub).fit()
        
            self.models[self.Label(**label)] = model

  
    # TODO I think this one can be cleaned up into a single 'apply' function
    def predict(self, test_data: pl.DataFrame):

        test_data = test_data.with_row_index(name='__row_index')

        subs = []
        for label in self.pomap.labels.iter_rows(named=True):
            sub = self.pomap.label_to_test(test_data, label=label)

            model = self.models[self.Label(**label)]

            predictions = model.predict(
                sub.to_pandas()
                ).rename('prediction')
            
            predictions = pl.from_pandas(predictions)
            
            sub = sub.with_columns(predictions).select('prediction', '__row_index')
            subs.append(sub)

        test_data = test_data.join(pl.concat(subs), on='__row_index', how='left').drop('__row_index')

        return test_data

In [29]:
ls = LiftedSmf(constructor, pomap)
ls.fit(train_df)

In [30]:
predictions = ls.predict(train_df)
predictions.filter(pl.col('cat2') == 'd')['prediction'].is_null().mean()

1.0

In [31]:
# Now test whether we can 'surgery' in a second model which trains unconditionally on cat2='d'
pomap2 = CategoricalPomap('cat2', ['d'])
ls2 = LiftedSmf(constructor, pomap2)
ls2.fit(train_df)

In [32]:
predictions2 = ls2.predict(train_df)
predictions2.filter(
    pl.col('cat2') == 'd'
)['prediction'].is_null().mean()

0.0